# Model 2 — YOLOv8l-P2 (small-object detector)

Adds a **P2 detection head** (stride-4, high-resolution) on top of YOLOv8l. The standard P3/P4/P5
heads down-sample a lot before detecting; P2 keeps a finer feature map, which is exactly what tiny
`crack` (~1% of image) and thin `scratch` need. 

There is no published `*-p2.pt`, so we **build the P2 architecture and transfer the COCO-pretrained
`yolov8l.pt` weights** into every matching layer (the new P2 head trains from scratch).

**Dataset:** CarDD (6 classes) — enhanced + class-balanced by `preprocess.ipynb`.
**Config:** `imgsz=1024`, **auto-batch** (`batch=-1`), **checkpoint every epoch** (`save_period=1`),
cosine LR, early stopping.



## 0 · GPU check — *fails if no CUDA*

In [1]:
import torch, sys, platform
assert torch.cuda.is_available(), "No CUDA GPU found. Activate the GPU env before running."
print("Python :", sys.version.split()[0], "|", platform.system())
print("Torch  :", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB VRAM")

Python : 3.10.20 | Windows
Torch  : 2.5.1+cu121 | CUDA 12.1
GPU    : NVIDIA GeForce RTX 4080 SUPER | 17.2 GB VRAM


## 1 · Imports

In [2]:
import os, shutil, yaml
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
import ultralytics
print("ultralytics", ultralytics.__version__)

ultralytics 8.4.66


## 2 · Resolve dataset paths (portable across machines)



In [3]:
HERE = Path.cwd()
ROOT = None
for cand in [HERE, *HERE.parents]:
    if (cand / "data_balanced.yaml").exists():
        ROOT = cand
        break
assert ROOT is not None, "Could not find data_balanced.yaml. Keep this notebook inside CarDD/model/."

with open(ROOT / "data_balanced.yaml") as f:
    dcfg = yaml.safe_load(f)
dcfg["path"] = str(ROOT).replace("\\", "/")

DATA_YAML = HERE / "data_local.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(dcfg, f, sort_keys=False, allow_unicode=True)

for sub in ([dcfg["train"]] if isinstance(dcfg["train"], str) else dcfg["train"]) + [dcfg["val"], dcfg["test"]]:
    p = ROOT / sub
    print(("  OK " if p.exists() else "  MISSING "), p)
print("\nUsing data config:", DATA_YAML)
print(open(DATA_YAML).read())

  OK  c:\Users\AFA001\Downloads\CarDD_transfer\enhanced\train\images
  OK  c:\Users\AFA001\Downloads\CarDD_transfer\aug_train\images
  OK  c:\Users\AFA001\Downloads\CarDD_transfer\enhanced\val\images
  OK  c:\Users\AFA001\Downloads\CarDD_transfer\enhanced\test\images

Using data config: c:\Users\AFA001\Downloads\CarDD_transfer\model\data_local.yaml
path: c:/Users/AFA001/Downloads/CarDD_transfer
train:
- enhanced/train/images
- aug_train/images
val: enhanced/val/images
test: enhanced/test/images
nc: 6
names:
  0: dent
  1: scratch
  2: crack
  3: glass_shatter
  4: lamp_broken
  5: tire_flat



## 3 · Training configuration

In [4]:
ARCH          = "yolov8l-p2.yaml"   # P2 architecture at 'l' scale (extra high-res head)
PRETRAINED    = "yolov8l.pt"        # COCO weights transferred into matching layers
RUN_NAME      = "02_yolov8l_p2"
PROJECT       = str(HERE / "runs")

TRAIN_ARGS = dict(
    data        = str(DATA_YAML),
    imgsz       = 1024,      # high res + P2 head = strongest small-object setup
    epochs      = 150,
    batch       = 0.85,        # AUTO-BATCH (P2 uses more VRAM, so the batch may be smaller)
    device      = 0,
    patience    = 30,
    save        = True,
    save_period = 1,         # CHECKPOINT EVERY EPOCH
    cos_lr      = True,
    close_mosaic= 15,
    fliplr      = 0.5,
    flipud      = 0.0,       # no vertical flip
    seed        = 42,
    plots       = True,
    val         = True,
    cache       = False,
    workers     = 8,
    optimizer   = "auto",
    amp         = True,
    project     = PROJECT,
    name        = RUN_NAME,
    exist_ok    = False,
)
for k, v in TRAIN_ARGS.items():
    print(f"  {k:12s}: {v}")

  data        : c:\Users\AFA001\Downloads\CarDD_transfer\model\data_local.yaml
  imgsz       : 1024
  epochs      : 150
  batch       : 0.85
  device      : 0
  patience    : 30
  save        : True
  save_period : 1
  cos_lr      : True
  close_mosaic: 15
  fliplr      : 0.5
  flipud      : 0.0
  seed        : 42
  plots       : True
  val         : True
  cache       : False
  workers     : 8
  optimizer   : auto
  amp         : True
  project     : c:\Users\AFA001\Downloads\CarDD_transfer\model\runs
  name        : 02_yolov8l_p2
  exist_ok    : False


## 4 · Build P2 model + transfer pretrained weights, then train

`YOLO(ARCH).load(PRETRAINED)` constructs the P2 network and copies every COCO-pretrained layer
whose shape matches (backbone + most of the neck); the new P2 head initializes fresh and learns
during training. Auto-batch sizes the run for the remote GPU automatically.


In [5]:
from pathlib import Path
from ultralytics import YOLO

# Use the already-trained P2 model (the good 0.533 run). NO further training.
RUNS_DIR = HERE / "runs"
def _epochs_in(d):
    csv = d / "results.csv"
    return (sum(1 for _ in open(csv)) - 1) if csv.exists() else 0

cands = [d for d in sorted(RUNS_DIR.glob("*"))
         if d.is_dir() and "02_yolov8l_p2" in d.name and "polish" not in d.name
         and (d / "weights" / "best.pt").exists()]
assert cands, "No trained P2 run with best.pt found under runs/."
SAVE_DIR = max(cands, key=_epochs_in)          # -> 02_yolov8l_p2-2 (your 46-epoch run)
model = YOLO(str(SAVE_DIR / "weights" / "best.pt"))

print("Using trained P2 model (no training):", SAVE_DIR.name)
print("Best weights:", SAVE_DIR / "weights" / "best.pt")


Using trained P2 model (no training): 02_yolov8l_p2-2
Best weights: c:\Users\AFA001\Downloads\CarDD_transfer\model\runs\02_yolov8l_p2-2\weights\best.pt


## 5 · Evaluate `best.pt` on the held-out **test** split

In [6]:
best = YOLO(str(SAVE_DIR / "weights" / "best.pt"))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=1024, plots=True)

print(f"\nTEST  mAP50-95: {metrics.box.map:.4f}")
print(f"TEST  mAP50   : {metrics.box.map50:.4f}")
print(f"TEST  mAP75   : {metrics.box.map75:.4f}")

names = metrics.names
rows = []
for i, c in enumerate(metrics.box.ap_class_index):
    rows.append({"class": names[c],
                 "AP50":     round(float(metrics.box.ap50[i]), 4),
                 "AP50-95":  round(float(metrics.box.ap[i]), 4),
                 "precision":round(float(metrics.box.p[i]), 4),
                 "recall":   round(float(metrics.box.r[i]), 4)})
per_class = pd.DataFrame(rows).sort_values("AP50-95")
display(per_class)
per_class.to_csv(SAVE_DIR / "test_per_class.csv", index=False)
print("Saved per-class metrics ->", SAVE_DIR / "test_per_class.csv")

Ultralytics 8.4.66  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
YOLOv8l-p2 summary (fused): 139 layers, 42,825,000 parameters, 0 gradients, 204.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 547.6165.5 MB/s, size: 241.0 KB)
val: Scanning C:\Users\AFA001\Downloads\CarDD_transfer\enhanced\test\labels.cache... 374 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 374/374  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.9it/s 8.3s0.4s
                   all        374        785      0.748      0.701      0.721      0.541
                  dent        157        236       0.68      0.517      0.577      0.323
               scratch        183        307      0.568      0.557      0.538      0.281
                 crack         48         70       0.53      0.443      0.471      0.255
         glass_shatter         71         71      0.939      0.972      0.982       0.

,class,AP50,AP50-95,precision,recall
2,crack,0.4712,0.2555,0.5303,0.4429
1,scratch,0.5377,0.2809,0.5683,0.5570
0,dent,0.5773,0.3226,0.6799,0.5169
4,lamp_broken,0.8344,0.6887,0.8570,0.7816
5,tire_flat,0.9258,0.8385,0.9134,0.9375
3,glass_shatter,0.9821,0.8596,0.9393,0.9718


Saved per-class metrics -> c:\Users\AFA001\Downloads\CarDD_transfer\model\runs\02_yolov8l_p2-2\test_per_class.csv


## 6 · Show training/result plots

In [9]:
from IPython.display import Image as IPyImage, display
for fn in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png", "val_batch0_pred.jpg"]:
    f = SAVE_DIR / fn
    if f.exists():
        print(fn); display(IPyImage(filename=str(f)))